> Notebook-friendly copy of `part-I/1.3-numpy.ipynb`, generated by `tools/make_live.py`. Edit the book notebook, not this file.

In [ ]:
# --- environment setup (generated, not part of the lesson) ---
# Colab and Kaggle start in an empty working directory.
# This is a no-op in an environment that is already set up.
from pathlib import Path

Path("_files").mkdir(exist_ok=True)   # the folder this notebook writes into

# 1.3) Scientific Computing with numpy

A Python `for` loop over a million temperature readings runs a million separate additions, each paying the interpreter's overhead; numpy replaces the loop with one array operation that runs in compiled code, often one to two orders of magnitude faster, and reads closer to the mathematical statement of the problem. That combination — speed and a notation that matches the science — is why numpy underlies the rest of the scientific Python stack, from pandas to scikit-learn to xarray. This notebook works with one running example, a small two-dimensional temperature field on a latitude–longitude grid, to cover array creation, dtype and shape, indexing and masking, vectorised math and broadcasting, reductions, and the handling of missing data — the same topics numpy's own [beginner documentation](https://numpy.org/doc/stable/user/absolute_beginners.html) covers, worth a look for a second pass at these ideas. It closes with a generated-code bug that silently truncates results because of a dtype mistake.

<img src="https://raw.githubusercontent.com/gse-unil/2026_MLEES_book/main/part-I/_static/numpy_logo_2020.png" alt="The NumPy project logo" width="500">

<em>The NumPy logo, from the project's own <a href="https://numpy.org/press-kit/">press kit</a>, made available for use in course materials.</em>

**🎯 Learning objectives**

- Create arrays from data and with constructors (`zeros`, `ones`, `arange`, `linspace`), and inspect their `shape` and `ndim`.
- Index, slice, and select elements with boolean masks.
- Replace element-wise loops with vectorised math and broadcasting.
- Call functions as a method (`arr.mean()`) or from numpy (`np.mean(arr)`).
- Reduce along chosen axes (`mean`, `min`, `max`, `argmax`, …), and reshape and stack arrays.
- Choose values with `np.where`, handle missing data with NaN-aware operations, and fill gaps with `np.interp`.
- Save and reload arrays with `np.save` and `np.load`.

You already know a container for a sequence of numbers: the `list`. An `ndarray` differs in
three ways that matter for scientific work.

| | `list` | `ndarray` |
| --- | --- | --- |
| Dimensions | one (nested lists to fake more) | any number, as a real shape |
| Contents | anything, mixed | one dtype for every element |
| Arithmetic | `*` repeats, `+` joins | element-wise maths |
| Speed | a Python loop per element | one compiled loop over the whole block |

The single dtype is what buys the speed: because every element has the same type and size,
numpy stores them in one contiguous block of memory and hands the loop to compiled code.

In [ ]:
import numpy as np

py_list = [1, 2, 3, 4]
arr = np.array([1, 2, 3, 4])

print(py_list * 2)     # [1, 2, 3, 4, 1, 2, 3, 4] — the list is repeated
print(arr * 2)         # [2 4 6 8] — every element is doubled

# a list can hold mixed types; an array cannot
print(type(1).__name__, type("two").__name__, type(3.0).__name__)   # int str float
print(np.array([1, 2, 3]).dtype)                                    # int64 — one type for all

## 1.3.1 Creating arrays

An array is created from data — a nested list — or from a constructor. Every array carries a `shape` (its size along each axis) and an `ndim` (the number of axes).

In [ ]:
import numpy as np

# a 2D field: 4 latitudes (rows) x 6 longitudes (cols), daily mean temp (°C)
temp_celsius = np.array([
    [ 5.2,  4.8,  6.1,  3.9,  2.7,  4.4],
    [ 1.3,  0.5, -0.8, -1.2,  0.9,  2.1],
    [-2.6, -3.1, -1.9,  0.2, -0.5,  1.1],
    [ 6.4,  7.0,  5.5,  8.1,  4.2,  5.8],
])

print(temp_celsius)
print("shape:", temp_celsius.shape, "| ndim:", temp_celsius.ndim)

<img src="https://raw.githubusercontent.com/gse-unil/2026_MLEES_book/main/part-I/_static/numpy_array_creation.png" alt="Diagram of the temp_celsius array with its shape and ndim labelled" width="500">

<em>`temp_celsius` as a grid of 4 rows by 6 columns: `shape` is the size along each axis, `ndim` is the number of axes.</em>

## 1.3.2 Other ways to create arrays

Beyond a literal, numpy provides constructors for the patterns you need most often.

In [ ]:
print(np.zeros((2, 3)))          # all zeros, given shape
print(np.ones(4))                # all ones
print(np.full((2, 2), 7.0))      # filled with a constant
print(np.arange(0, 10, 2))       # evenly spaced by step: [0 2 4 6 8]
print(np.linspace(0.0, 1.0, 5))  # n evenly spaced points: [0. 0.25 0.5 0.75 1. ]
print(np.random.random(3))       # 3 random floats in [0, 1)

<details>
<summary><b>🔍 Going deeper: dtype and precision</b></summary>

Every array also has a `dtype` — the element type — which fixes both its behaviour and its memory use. `float64` (double precision) is the default; `float32` halves the memory at the cost of precision.

```python
temp32 = temp_celsius.astype(np.float32)
print(temp_celsius.dtype, temp_celsius.nbytes, "bytes")   # float64, 192 bytes
print(temp32.dtype, temp32.nbytes, "bytes")               # float32, 96 bytes

# float32 is coarser: the rounding error in 0.1 + 0.2 disappears
print(np.float64(0.1) + np.float64(0.2))   # 0.30000000000000004
print(np.float32(0.1) + np.float32(0.2))   # 0.3
```

Watch the dtype when preallocating output arrays with `np.zeros_like`: it silently copies the input's dtype, so an integer input produces an integer output even when the result should be fractional.

</details>

Coordinates often come as two 1D axes, but a field defined on the grid needs a value at every
(row, column) pair. `meshgrid` expands the two axes into two 2D arrays of matching shape — one
holding the longitude of each cell, the other its latitude. You will use these constantly when
plotting fields in the next subchapter.

In [ ]:
lon = np.linspace(6.0, 9.0, 6)     # 6 longitudes
lat = np.linspace(46.0, 47.5, 4)   # 4 latitudes

lon2d, lat2d = np.meshgrid(lon, lat)
print(lon2d.shape, lat2d.shape)    # (4, 6) (4, 6) — one value per grid cell
print(lon2d)

## 1.3.3 Indexing, slicing, and boolean masking

Indexing uses `[row, col]`; slicing selects sub-blocks; negative indices count from the end. A boolean *mask* is a same-shaped array of True/False that selects the matching elements. Basic slicing (with `:`) returns a *view*, not a copy: it shares memory with the original array, so assigning into a slice also changes the array it was sliced from.

In [ ]:
print(temp_celsius[0, 0])      # one element
print(temp_celsius[0, :])      # first row, all longitudes
print(temp_celsius[:, -1])     # last column, all latitudes
print(temp_celsius[1:3, 2:4])  # a 2x2 sub-block

# a boolean mask and what it selects
freezing = temp_celsius < 0.0
print("n freezing cells:", int(freezing.sum()))    # True counts as 1
print("freezing values:", temp_celsius[freezing])  # 1D of matches

<img src="https://raw.githubusercontent.com/gse-unil/2026_MLEES_book/main/part-I/_static/numpy_indexing_4panel.png" alt="Diagram of four indexing and slicing operations on a 4 by 6 array" width="700">

<em>Four ways to select from `temp_celsius`: a single element, a full row, a full column, and a 2x2 sub-block.</em>

<img src="https://raw.githubusercontent.com/gse-unil/2026_MLEES_book/main/part-I/_static/numpy_boolean_mask.png" alt="Diagram of a boolean mask, the cells it selects, and the resulting 1D array" width="2000">

<em>`temp_celsius < 0.0` produces a same-shaped array of `True`/`False`; indexing with it, `temp_celsius[freezing]`, pulls out the matching cells as a flat 1D array — not a row or a column, since the selected cells no longer form a rectangle.</em>

**🧠 Computational-thinking fundamental: think in arrays, not loops**

The central idea of numpy is *vectorisation*: express a computation as an operation on whole arrays rather than a loop over elements. `field + 273.15` converts every value at once. Vectorised code is far faster, since the loop runs in compiled C instead of the Python interpreter — and it reads closer to the mathematical statement of the problem, with no loop to obscure it. When you find yourself writing a Python `for` loop over array elements, look for the array operation that replaces it.

<details>
<summary><b>🔍 Going deeper: memory layout, views, and strides</b></summary>

An array is a flat block of memory plus a `shape` and `strides` (the byte step along each axis). Slicing returns a *view* that shares memory with the original, so writing through it mutates the source; use `.copy()` for an independent array. C-order (row-major, the default) and Fortran-order (column-major) change which axis is contiguous and therefore which traversals are fastest.

```python
a = np.arange(12).reshape(3, 4)
print(a.strides)          # bytes to step along (rows, cols)
b = a[:, 1]               # a view, not a copy
b[0] = 999                # this also changes a[0, 1]
```

</details>

## 1.3.4 Vectorised math and broadcasting

A single expression applies element-wise across an array. *Broadcasting* lets arrays of different but compatible shapes combine: a length-4 column vector can be stretched across all 6 longitudes.

In [ ]:
# vectorised: no python loop
temp_kelvin = temp_celsius + 273.15
print(temp_kelvin)  # shape (4,6)

<img src="https://raw.githubusercontent.com/gse-unil/2026_MLEES_book/main/part-I/_static/numpy_vectorized_add.png" alt="Diagram of temp_celsius plus a scalar producing temp_kelvin, with no loop" width="700">

<em>`temp_celsius + 273.15` updates every cell in one expression — no `for` loop needed.</em>

Broadcasting aligns shapes from the **last** axis backwards. Our field is `(4, 6)` and the latitude gradient is `(4,)` — numpy tries to match the 4 against the 6, fails, and raises an error. We have to say explicitly that the gradient varies along the *rows*, by giving it a second axis of length 1.

Indexing with `None` inserts a new axis at that position, turning a `(4,)` vector into a `(4, 1)` column. `np.newaxis` is the same thing spelled out, and you will see both.

In [ ]:
lat_gradient_celsius = np.array([0.0, -1.5, -3.0, -4.5])   # colder toward the north

print(lat_gradient_celsius.shape)                  # (4,)   — a flat vector
print(lat_gradient_celsius[:, None].shape)         # (4, 1) — now a column
print(lat_gradient_celsius[:, np.newaxis].shape)   # (4, 1) — the same thing

In [ ]:
# without the extra axis, the shapes cannot be aligned:
# temp_celsius + lat_gradient_celsius
# ValueError: operands could not be broadcast together with shapes (4,6) (4,)

<img src="https://raw.githubusercontent.com/gse-unil/2026_MLEES_book/main/part-I/_static/numpy_broadcast_impossible.png" alt="Diagram of temp_celsius (4, 6) and lat_gradient_celsius (4,) failing to broadcast" width="700">

<em>Broadcasting aligns shapes from the last axis backwards: 6 against 4 does not match, so `temp_celsius + lat_gradient_celsius` raises a `ValueError`.</em>

In [ ]:
# broadcasting: a (4,1) column stretches across the 6 longitudes
lat_gradient_celsius = np.array([0.0, -1.5, -3.0, -4.5])   # colder toward the north
adjusted = temp_celsius + lat_gradient_celsius[:, None]    # (4,1) + (4,6) -> (4,6)
print(adjusted.shape)
print(adjusted)

<img src="https://raw.githubusercontent.com/gse-unil/2026_MLEES_book/main/part-I/_static/numpy_broadcast_result.png" alt="Diagram of the (4, 1) column broadcasting across the 6 columns of temp_celsius to produce adjusted" width="900">

<em>The (4, 1) column stretches across all 6 longitudes to combine with the (4, 6) field. (In linear-algebra notation this is `A + b·1ᵀ`: the column `b` repeated across every column via an outer product with a row of ones.)</em>

<details>
<summary><b>🔍 Going deeper: vectorisation vs loops</b></summary>

Vectorised array operations run in compiled code and are typically one to two orders of magnitude faster than an equivalent Python loop. You can measure it in a notebook:

```python
big = np.random.random(1_000_000)
%timeit big + 1.0                       # vectorised
%timeit [x + 1.0 for x in big]          # Python loop, much slower
```

The gap widens with array size. Reach for the array expression first; drop to a loop only when no vectorised form exists.

</details>

## 1.3.5 Calling functions on arrays

Most numpy operations are available two ways: as a *method* on the array (`arr.mean()`) or as a *function* in the numpy namespace (`np.mean(arr)`). They do the same thing — pick whichever reads more clearly.

In [ ]:
print(temp_celsius.mean(), np.mean(temp_celsius))   # method and function agree
print(temp_celsius.sum(), np.sum(temp_celsius))

<img src="https://raw.githubusercontent.com/gse-unil/2026_MLEES_book/main/part-I/_static/numpy_mean_sum.png" alt="Diagram of temp_celsius collapsing to a single mean value and a single sum value" width="700">

<em>`.mean()` and `.sum()` with no `axis` collapse the whole array to one scalar.</em>

<details>
<summary><b>🔍 Going deeper: other useful numpy functions</b></summary>

numpy provides element-wise maths and helpers for tidying values. A few you will reach for often:

```python
a = np.array([1.234, -2.5, 9.876])
print(a.round(2))          # round to 2 decimals: [ 1.23 -2.5   9.88]
print(np.abs(a))           # absolute value
print(np.sqrt([1, 4, 9]))  # element-wise square root: [1. 2. 3.]
print(np.clip(a, 0, 5))    # limit values to the range [0, 5]
print(np.allclose([1.0, 2.0], [1.0, 2.0000001]))   # True — element-wise float comparison
```

`round` is useful for readable output, but it is only for display — keep full precision inside a calculation. `np.allclose` is the array version of the `math.isclose` you met in 1.1: never compare float
arrays with `==`.

</details>

## 1.3.6 Reductions, reshape, and stacking

A reduction collapses an axis: `axis=0` aggregates over latitudes (one result per longitude), `axis=1` over longitudes. `reshape` reorganises the same data; stacking combines arrays.

Common reductions all take an optional `axis`:

| Function | Returns |
| --- | --- |
| `sum`, `mean`, `std` | total, average, spread |
| `min`, `max` | smallest / largest value |
| `argmin`, `argmax` | index of the smallest / largest value |
| `cumsum`, `cumprod` | running total / product |

In [ ]:
# a reduction collapses an axis: axis=0 over rows, axis=1 over columns
print("overall mean:", temp_celsius.mean())
print("mean per column (over rows):", temp_celsius.mean(axis=0))
print("mean per row (over columns):", temp_celsius.mean(axis=1))
print("min, max:", temp_celsius.min(), temp_celsius.max())
print("argmin, argmax (flat index):", temp_celsius.argmin(), temp_celsius.argmax())

# reshape: same 24 values, new shape; -1 infers the missing length
flat = temp_celsius.reshape(-1)
print("reshaped:", flat.shape)

# stacking: combine arrays along a new row axis
col_means = temp_celsius.mean(axis=0)
index_row = np.arange(6, dtype=float)
print("vstacked:", np.vstack([index_row, col_means]).shape)

<img src="https://raw.githubusercontent.com/gse-unil/2026_MLEES_book/main/part-I/_static/numpy_reductions_axis.png" alt="Diagram of temp_celsius reduced along axis 0 to col_means and along axis 1 to row_means" width="800">

<em>`mean(axis=0)` collapses the 4 rows to one value per column; `mean(axis=1)` collapses the 6 columns to one value per row (shown here as a row too, for comparison).</em>

<img src="https://raw.githubusercontent.com/gse-unil/2026_MLEES_book/main/part-I/_static/numpy_reshape.png" alt="Diagram of temp_celsius reshaped from (4, 6) into a flat (24,) array" width="800">

<em>`reshape(-1)` reorganises the same 24 values into one flat row; nothing is recomputed.</em>

<img src="https://raw.githubusercontent.com/gse-unil/2026_MLEES_book/main/part-I/_static/numpy_vstack.png" alt="Diagram of index_row and col_means stacked into a (2, 6) array" width="500">

<em>`np.vstack([index_row, col_means])` stacks two (6,) rows into one (2, 6) array.</em>

**ℹ️ Quick exercise: warmest latitude**

Compute the mean temperature of each latitude (each row), then use `argmax` to find the index of the warmest latitude.


<details>
<summary><b>✅ Solution</b></summary>

```python
row_means = temp_celsius.mean(axis=1)
print(row_means)
print(int(row_means.argmax()))
```

</details>

<details>
<summary><b>🔍 Going deeper: basic linear algebra</b></summary>

numpy covers the everyday linear algebra a model needs.

```python
A = np.array([[2.0, 1.0], [1.0, 3.0]])
b = np.array([1.0, 2.0])
print(A @ b)                 # matrix-vector product (also np.matmul)
print(np.linalg.solve(A, b)) # solve A x = b without inverting A
```

Prefer `np.linalg.solve` over forming `np.linalg.inv(A) @ b`: it is more accurate and faster.

</details>

## 1.3.7 Choosing, missing data, and interpolation

`np.where` picks element-wise between two options. Missing data is represented by `NaN`; `np.isnan` flags which elements are missing, NaN-aware reductions (`np.nanmean`, …) skip them, and ordinary reductions propagate `NaN` instead. `np.interp` fills a 1D gap by linear interpolation.

In [ ]:
# np.where(condition, a, b): element-wise choice
category = np.where(temp_celsius < 0.0, "freezing", "above")
print(category)

In [ ]:
# missing data as NaN; nan-aware vs ordinary reduction
temp_with_gaps = temp_celsius.copy()
temp_with_gaps[0, 0] = np.nan
print("nanmean (skips gaps):", np.nanmean(temp_with_gaps))
print("plain mean is contaminated:", np.mean(temp_with_gaps))  # nan
print("n missing:", np.isnan(temp_with_gaps).sum())      # np.isnan flags them; True counts as 1

In [ ]:
# np.interp: fill a 1D gap by linear interpolation against an index
profile = temp_celsius[:, 0].copy()      # the first column, 4 latitudes
x = np.arange(profile.size)
known = np.array([0, 1, 3])              # pretend index 2 is missing
filled = np.interp(x, known, profile[known])
print(profile)
print(filled)                            # index 2 interpolated from its neighbours

<img src="https://raw.githubusercontent.com/gse-unil/2026_MLEES_book/main/part-I/_static/numpy_where_nan.png" alt="Diagram of np.where categorizing cells and a NaN marking a missing cell" width="700">

<em>`np.where(temp_celsius < 0, "freezing", "above")` labels every cell; a `NaN` marks a missing one, propagating through an ordinary `.mean()` but skipped by `np.nanmean()`.</em>

## 1.3.8 Saving and loading arrays

numpy can write an array to disk in its own binary format, `.npy`, and read it back exactly as
it was — shape, dtype, and all. This is handy for intermediate results.

In [ ]:
from pathlib import Path
Path("_files").mkdir(exist_ok=True)      # _files/ is not committed, so create it first

np.save("_files/temp_field.npy", temp_celsius)          # writes temp_field.npy

reloaded = np.load("_files/temp_field.npy")
print(reloaded.shape, reloaded.dtype)
print(np.allclose(temp_celsius, reloaded))       # True — an exact round trip

**ℹ️ .npy is for convenience, not for archiving**

`.npy` files are readable only by numpy and carry no metadata: no units, no coordinates, no
description of what the numbers mean. They are fine for temporary results inside one project.
For data you intend to keep or share, use a self-describing format such as netCDF — introduced
with xarray in the next subchapter.

<details>
<summary><b>🔍 Going deeper: bigger-than-memory arrays with dask</b></summary>

When a field is too large for memory, `dask.array` exposes the same numpy interface over *chunks*, building a task graph that runs only when you call `.compute()`.

```python
import dask.array as da
x = da.from_array(np.arange(1_000_000), chunks=100_000)
result = (x + 1).mean()      # lazy: nothing computed yet
print(result.compute())      # runs the graph, chunk by chunk
```

xarray uses this same lazy, chunked model for climate-scale datasets in later subchapters.

</details>

## *When generated code lies: a silent dtype truncation*

AI assistants often preallocate an output array with `np.zeros_like`, which copies the *input's* dtype. If the input is integer, float results are silently truncated on assignment. Here a function computes anomalies (value minus the field mean) for an integer precipitation field.

In [ ]:
def to_anomaly(field):
    # subtract the mean into a preallocated array (as an assistant returned it)
    result = np.zeros_like(field)        # inherits field's dtype!
    result[:] = field - field.mean()
    return result

precip_mm = np.array([[0, 2, 5], [1, 0, 8], [3, 4, 2]])   # integer mm
print("input dtype:", precip_mm.dtype)
print(to_anomaly(precip_mm))

**⚠️ Diagnosis: the output inherited an integer dtype**

The anomalies should be fractional, but every value is a whole number. `np.zeros_like(field)` produced an *integer* array because `field` is integer, and assigning float anomalies into it truncates each toward zero. The error is silent — no exception, just wrong numbers. The fix is to let numpy choose the result type, or to request `dtype=float` explicitly.

In [ ]:
def to_anomaly(field):
    # let numpy promote to float; no wrong-dtype preallocation
    return field - field.mean()

print(to_anomaly(precip_mm))
print("output dtype:", to_anomaly(precip_mm).dtype)

**📌 Takeaways**

- Broadcasting aligns shapes from the last axis backwards; add a length-1 axis with `[:, None]` when you need a vector to vary along the rows.
- Index and slice with `[row, col]`; select with boolean masks.
- Vectorise: write array expressions instead of element loops, and use broadcasting to combine compatible shapes; most operations exist as both a method (`arr.mean()`) and an `np.` function (`np.mean(arr)`).
- Reduce along an explicit `axis` (`mean`, `sum`, `min`, `max`, `argmin`, `argmax`); reshape and stack to reorganise data.
- Use `np.where` to choose element-wise, `np.isnan` and NaN-aware ops for missing data, and `np.interp` to fill 1D gaps.
- Watch dtype on preallocation: `np.zeros_like(int_array)` truncates float results silently — let numpy promote to float.
- *(going deeper)* Every array has a dtype (`float64` default, `float32` half the size and coarser); slicing returns a view that shares memory with the original, so copy with `.copy()` when you need independence.

## Summary

| Concept | Rule to remember |
|---|---|
| Arrays | One dtype and one shape; build them with `np.array`, `zeros`, `ones`, `arange`, `linspace`. |
| Indexing | `[row, col]` selects, boolean masks filter. |
| Vectorising | Write array expressions instead of element-by-element loops. |
| Broadcasting | Shapes align from the last axis backwards; `[:, None]` adds a length-1 axis. |
| Reductions | State the `axis` explicitly (`mean`, `sum`, `min`, `max`, `argmin`, `argmax`). |
| Missing data | `np.where` chooses element-wise, `np.isnan` detects, `np.interp` fills 1D gaps. |
| dtype | `np.zeros_like(int_array)` truncates float results silently — let numpy promote to float. |

## Resources

- [NumPy: the absolute basics for beginners](https://numpy.org/doc/stable/user/absolute_beginners.html) — numpy's own official introduction, covering the same array-creation, indexing, and broadcasting ideas as this notebook.
- [Python Data Science Handbook — Introduction to NumPy](https://jakevdp.github.io/PythonDataScienceHandbook/02.00-introduction-to-numpy.html) — free online; thorough coverage of arrays, broadcasting, masking, and ufuncs.
- [Scientific Python Lectures — NumPy](https://lectures.scientific-python.org/intro/numpy/index.html) — a concise, research-oriented tour of array creation, operations, and reductions.